In [0]:
%run /Workspace/Users/valterlafuentejunior@gmail.com/IngestaodedadosAPInoDatabricks/Config/config

In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession

In [0]:
def buscar_dados_acao(ticker_b3, api_key):
    """
    Função para buscar dados diários de ações da B3 usando a API da Alpha Vantage.

    Parâmetros:
    ----------
    ticker_b3 : str
        Código da ação na B3 (exemplo: 'PETR4', 'VALE3', 'ITUB4').
    api_key : str
        Chave de acesso (API Key) obtida no site da Alpha Vantage.

    Retorno:
    -------
    pandas.DataFrame ou None
        Retorna um DataFrame com os dados diários da ação se a consulta for bem-sucedida.
        Retorna None caso ocorra erro ou não haja dados disponíveis.
    """

    # 🔹 Concatena o sufixo .SA (para indicar que é ação da B3)
    ticker = ticker_b3 + ".SA"

    # 🔹 URL base da API
    url = "https://www.alphavantage.co/query"

    # 🔹 Parâmetros da requisição
    params = {
        "function": "TIME_SERIES_DAILY",  # Tipo de dado (série temporal diária)
        "symbol": ticker,                 # Código da ação
        "apikey": api_key,                # Chave da API
        "outputsize": "compact"           # Pode ser 'compact' (últimos 100 dias) ou 'full'
    }

    #  Faz a requisição HTTP
    response = requests.get(url, params=params)

    #  Verifica se a requisição foi bem-sucedida
    if response.status_code != 200:
        print(f"[{ticker_b3}] Erro HTTP: {response.status_code}")
        return None

    #  Converte a resposta para JSON
    data = response.json()
    print("MEU DATA", data)

    #  Verifica se o campo 'Time Series (Daily)' está presente
    if "Time Series (Daily)" not in data:
        print(f"[{ticker_b3}] Sem dados ou limite de uso da API atingido.")
        return None

    #  Converte o JSON em DataFrame
    df = pd.DataFrame.from_dict(data["Time Series (Daily)"], orient="index")

    #  Renomeia as colunas para nomes mais amigáveis
    df.columns = ["Abertura", "Alta", "Baixa", "Fechamento", "Volume"]

    df = df.astype(float)

    #  Converte o índice (datas) para o formato datetime
    df.index = pd.to_datetime(df.index)

    #  Ordena as datas em ordem crescente
    df.sort_index(inplace=True)

    df["ticker"] = ticker_b3

    df["data_ingestao"] = datetime.now()

    return df.reset_index().rename(columns={"index": "data"})

In [0]:

# ======================================================
# 🔹 Coleta de dados para todos os tickers
# ======================================================
dfs = []
for ticker in tickets:
    df = buscar_dados_acao(ticker, API_KEY)
    if df is not None:
        print(f"✅ Ticker {ticker} coletado com sucesso!")
        dfs.append(df)
    else:
        print(f"⚠️ Falha ao coletar dados do ticker {ticker}")

# ======================================================
# 🔹 Criação do DataFrame final e gravação no Delta Lake
# ======================================================
if dfs:
    # Concatena todos os DataFrames coletados
    df_final = pd.concat(dfs, ignore_index=True)
    df_spark = spark.createDataFrame(df_final)

    # ==================================================
    # 🔧 Criação automática do schema e da tabela Delta
    # ==================================================
    print("\n🔧 Verificando schema e tabela no Unity Catalog...")

    # Garante que o schema 'bronze' exista
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS workspace.bronze
        COMMENT 'Camada Bronze para armazenamento de dados brutos'
    """)

    # Verifica se a tabela já existe
    tabela_existe = spark.catalog.tableExists("workspace.bronze.cotacoes")

    # Cria a tabela se não existir, ou adiciona dados se já existir
    if not tabela_existe:
        print("🆕 Tabela 'workspace.bronze.cotacoes' não encontrada. Criando agora...")
        (
            df_spark.write
            .format("delta")
            .mode("overwrite")  # Apenas na primeira execução
            .saveAsTable("workspace.bronze.cotacoes")
        )
        print("✅ Tabela criada com sucesso!")
    else:
        print("📥 Tabela existente detectada. Inserindo novos dados (append)...")
        (
            df_spark.write
            .format("delta")
            .mode("append")  # Demais execuções
            .saveAsTable("workspace.bronze.cotacoes")
        )
        print("✅ Dados adicionados com sucesso!")

    # ==================================================
    # 🔍 Validação final (opcional)
    # ==================================================
    print("\n📊 Amostra dos dados gravados:")
    spark.sql("SELECT * FROM workspace.bronze.cotacoes LIMIT 5").show(truncate=False)

else:
    print("⚠️ Nenhum DataFrame foi coletado. Nenhum dado será gravado.")
